In [1]:
# pip install "tensorflow==2.15.1" "keras==2.15.0" wandb ipykernel

In [2]:
# ! pip install xgboost

In [3]:
# !pip install torch 

In [4]:
# pip install scikit-learn

In [5]:
import os
import wandb
from tensorflow import keras
from tensorflow.keras import layers

In [6]:
wandb.login()

wandb: Currently logged in as: nikhilanil2609 (nikhilanil2609-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

run = wandb.init(
    project="sentiment-analysis-nlp",
    name="lstm-imdb-sentiment",
    config={
        "learning_rate": 0.003,
        "epochs": 15,
        "batch_size": 32,
        "hidden_size": 128,
        "num_layers": 2,
        "embedding_dim": 100,
        "vocab_size": 5000,
        "architecture": "LSTM"
    }
)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab_size=5000, max_len=200):
        self.vectorizer = CountVectorizer(max_features=vocab_size)
        self.data = []
        self.labels = labels
   
        for text in texts:
            tokens = text.lower().split()[:max_len]
            # Convert to indices (simplified)
            indices = [hash(word) % vocab_size for word in tokens]
            # Pad sequences
            if len(indices) < max_len:
                indices += [0] * (max_len - len(indices))
            self.data.append(indices)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, output_size=2):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers, 
                           batch_first=True, dropout=0.3, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, 64)
        self.dropout = nn.Dropout(0.5)
        self.output = nn.Linear(64, output_size)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Use last hidden state
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        fc_out = self.relu(self.fc(hidden))
        fc_out = self.dropout(fc_out)
        out = self.output(fc_out)
        return out
print("Loading data...")
np.random.seed(42)
positive_reviews = ["great movie love it", "excellent film amazing", "wonderful fantastic", "best ever seen"] * 250
negative_reviews = ["terrible awful bad", "worst movie ever", "horrible waste time", "disappointing poor"] * 250

texts = positive_reviews + negative_reviews
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

train_dataset = SentimentDataset(train_texts, train_labels, vocab_size=wandb.config.vocab_size)
val_dataset = SentimentDataset(val_texts, val_labels, vocab_size=wandb.config.vocab_size)

train_loader = DataLoader(train_dataset, batch_size=wandb.config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=wandb.config.batch_size, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SentimentLSTM(
    vocab_size=wandb.config.vocab_size,
    embedding_dim=wandb.config.embedding_dim,
    hidden_size=wandb.config.hidden_size,
    num_layers=wandb.config.num_layers
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=wandb.config.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

wandb.watch(model, criterion, log="all", log_freq=50)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / len(loader), 100. * correct / total, all_preds, all_labels

print("Starting training...")
best_val_acc = 0

for epoch in range(wandb.config.epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, device)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Log metrics
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "learning_rate": current_lr
    })
    
    # Log confusion matrix every 5 epochs
    if (epoch + 1) % 5 == 0:
        wandb.log({
            "confusion_matrix": wandb.plot.confusion_matrix(
                probs=None,
                y_true=val_labels,
                preds=val_preds,
                class_names=["Negative", "Positive"]
            )
        })
    
    print(f'Epoch {epoch+1}/{wandb.config.epochs}:')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    print(f'  Learning Rate: {current_lr:.6f}')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_sentiment_model.pth')
        wandb.run.summary["best_val_accuracy"] = best_val_acc

print("\nFinal Evaluation...")
_, final_acc, _, _ = validate(model, val_loader, criterion, device)
run.summary["final_accuracy"] = final_acc

artifact = wandb.Artifact('sentiment-lstm-model', type='model')
artifact.add_file('best_sentiment_model.pth')
run.log_artifact(artifact)

print(f'\nBest Validation Accuracy: {best_val_acc:.2f}%')
run.finish()

Loading data...
Starting training...
Epoch 1/15:
  Train Loss: 0.0803, Train Acc: 97.25%
  Val Loss: 0.0000, Val Acc: 100.00%
  Learning Rate: 0.003000
Epoch 2/15:
  Train Loss: 0.0000, Train Acc: 100.00%
  Val Loss: 0.0000, Val Acc: 100.00%
  Learning Rate: 0.003000
Epoch 3/15:
  Train Loss: 0.0000, Train Acc: 100.00%
  Val Loss: 0.0000, Val Acc: 100.00%
  Learning Rate: 0.003000
Epoch 4/15:
  Train Loss: 0.0001, Train Acc: 100.00%
  Val Loss: 0.0000, Val Acc: 100.00%
  Learning Rate: 0.001500
Epoch 5/15:
  Train Loss: 0.0000, Train Acc: 100.00%
  Val Loss: 0.0000, Val Acc: 100.00%
  Learning Rate: 0.001500
Epoch 6/15:
  Train Loss: 0.0000, Train Acc: 100.00%
  Val Loss: 0.0000, Val Acc: 100.00%
  Learning Rate: 0.001500
Epoch 7/15:
  Train Loss: 0.0000, Train Acc: 100.00%
  Val Loss: 0.0000, Val Acc: 100.00%
  Learning Rate: 0.000750
Epoch 8/15:
  Train Loss: 0.0000, Train Acc: 100.00%
  Val Loss: 0.0000, Val Acc: 100.00%
  Learning Rate: 0.000750
Epoch 9/15:
  Train Loss: 0.0000, Tr

wandb: ERROR The nbformat package was not found. It is required to save notebook history.



Best Validation Accuracy: 100.00%


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
learning_rate,███▄▄▄▂▂▂▁▁▁▁▁▁
train_accuracy,▁██████████████
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_accuracy,100
epoch,15
final_accuracy,100
learning_rate,0.00019
train_accuracy,100
